<a href="https://colab.research.google.com/github/Heng1222/MalSem-Decon/blob/main/0504.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0504 社團名稱 Embedding 與 UMAP 3D 視覺化

這份 notebook 會讀取 `Data/nodes_page_1hop.csv` 的 `name` 欄位，使用 `ckiplab/bert-base-chinese` 產生社團名稱 embedding，並建立兩種 UMAP 3D 對照圖：二元惡意標記，以及「黃、賭、詐欺、借貸、多重、正常」六類標記。

In [11]:
# 若缺少套件，先取消註解並執行下一行：
%pip install pandas numpy torch transformers umap-learn plotly tqdm

from pathlib import Path

import pandas as pd

DATA_PATH = Path("Data/nodes_page_1hop.csv")
NEGATIVE_KEYWORDS_PATH = Path("Data/negative_keywords.txt")
GROUP_NAME_COLUMN = "name"

nodes = pd.read_csv(
    DATA_PATH,
    usecols=[GROUP_NAME_COLUMN],
    dtype={GROUP_NAME_COLUMN: "string"},
    encoding="utf-8-sig",
)

group_names = nodes[GROUP_NAME_COLUMN].fillna("").astype(str).str.strip()
group_names = group_names[group_names.ne("")].reset_index(drop=True)

print(f"社團名稱數量: {len(group_names):,}")
display(pd.DataFrame({"社團名稱": group_names.head(10)}))

社團名稱數量: 2,030,756


,社團名稱
0,親妮o'lens 代購
1,Uber
2,柯夢波丹Cosmopolitan Taiwan
3,高以翔 Godfrey Gao
4,水晶8號星球
5,ROCKCOCO
6,FM 你的試鞋間
7,Dr.Hojyo北条博士
8,英諾華隱形眼鏡
9,Homita


In [ ]:
import hashlib

import numpy as np
import plotly.express as px
import torch
import umap
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "ckiplab/bert-base-chinese"
RANDOM_STATE = 42
BATCH_SIZE = 64
MAX_LENGTH = 32

# 若資料量太大、只是要先確認流程，可改成整數，例如 20000；None 代表使用全部社團名稱。
MAX_GROUPS = None

if MAX_GROUPS is None:
    work_names = group_names.reset_index(drop=True)
else:
    work_names = group_names.sample(
        n=min(MAX_GROUPS, len(group_names)), random_state=RANDOM_STATE
    ).reset_index(drop=True)

def load_keywords(path: Path) -> list[str]:
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]

def keyword_hits(text: str, keywords: list[str]) -> list[str]:
    text_norm = str(text).casefold()
    return [kw for kw in keywords if kw.casefold() in text_norm]

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

@torch.inference_mode()
def encode_texts(texts: list[str], batch_size: int = BATCH_SIZE) -> np.ndarray:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()

    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding group names"):
        batch = texts[start : start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        batch_embeddings = mean_pooling(output, encoded["attention_mask"])
        vectors.append(batch_embeddings.cpu().numpy())

    return np.vstack(vectors).astype("float32")

def names_digest(names: pd.Series) -> str:
    joined = "\n".join(names.astype(str).tolist())
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:16]

cache_key = f"{MODEL_NAME.replace('/', '_')}_{len(work_names)}_{names_digest(work_names)}"
embedding_cache_path = Path("Data") / f"group_name_embeddings_{cache_key}.npy"

if embedding_cache_path.exists():
    embeddings = np.load(embedding_cache_path)
    print(f"讀取 embedding cache: {embedding_cache_path}")
else:
    embeddings = encode_texts(work_names.astype(str).tolist())
    np.save(embedding_cache_path, embeddings)
    print(f"已儲存 embedding cache: {embedding_cache_path}")

n_neighbors = min(15, max(2, len(work_names) - 1))
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=n_neighbors,
    min_dist=0.1,
    metric="cosine",
    random_state=RANDOM_STATE,
)
coords = reducer.fit_transform(embeddings)

negative_keywords = load_keywords(NEGATIVE_KEYWORDS_PATH)
negative_hits = [keyword_hits(name, negative_keywords) for name in work_names]

binary_viz_df = pd.DataFrame(
    {
        "社團名稱": work_names,
        "umap_x": coords[:, 0],
        "umap_y": coords[:, 1],
        "umap_z": coords[:, 2],
        "類別": ["惡意社團" if hits else "一般社團" for hits in negative_hits],
        "命中關鍵詞": ["、".join(hits) for hits in negative_hits],
    }
)

display(binary_viz_df["類別"].value_counts().rename_axis("類別").to_frame("數量"))

fig = px.scatter_3d(
    binary_viz_df,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="類別",
    color_discrete_map={"惡意社團": "#d62728", "一般社團": "#1f77b4"},
    hover_data={"社團名稱": True, "命中關鍵詞": True, "umap_x": False, "umap_y": False, "umap_z": False},
    title="ckiplab/bert-base-chinese 社團名稱 Embedding：惡意 / 一般社團 UMAP 3D",
)
fig.update_traces(marker={"size": 3, "opacity": 0.78})
fig.update_layout(legend_title_text="類別", margin={"l": 0, "r": 0, "t": 48, "b": 0})
fig.show()

In [2]:
concept_keywords = {
    "黃": [
        "黃", "色情", "情色", "成人", "成人片", "AV", "女優", "約炮", "援交", "外約",
        "外送茶", "茶莊", "茶魚", "半套", "全套", "按摩", "舒壓", "個工", "傳播",
        "禮服店", "三溫暖", "制服店", "裸聊", "私密", "辣妹", "妹子", "直播",
        "寫真", "情趣", "酒店", "八大", "夜場", "招待所",
    ],
    "賭": [
        "賭", "賭博", "博弈", "博奕", "娛樂城", "運彩", "彩券", "彩票", "百家樂",
        "德州", "撲克", "麻將", "老虎機", "輪盤", "下注", "投注", "賠率", "盤口",
        "球版", "賽馬", "六合彩", "今彩", "威力彩", "刮刮樂",
    ],
    "詐欺": [
        "詐", "詐騙", "詐欺", "車手", "人頭帳戶", "人頭戶", "洗錢", "釣魚", "盜刷",
        "刷單", "假投資", "投資群", "飆股", "飆股群", "內線", "老師帶單", "帶單",
        "代操", "保證獲利", "穩賺", "高報酬", "高獲利", "致富", "炫富", "套利",
        "虛擬貨幣", "加密貨幣", "幣圈", "USDT", "泰達幣", "出金", "入金", "抽成",
    ],
    "借貸": [
        "貸款", "信貸", "快易貸", "企業貸", "借貸", "借款", "借錢", "小額", "融資",
        "代書", "當舖", "二胎", "房貸", "車貸", "債務", "債務整合", "破產", "法拍",
        "卡債", "信用瑕疵", "強力過件", "免頭期", "全額貸", "全額", "協商", "清算",
        "週轉", "急用錢", "現金", "日領", "不問經歷", "槓桿", "增貸",
    ],
}

def classify_concepts(text: str) -> tuple[str, list[str], list[str]]:
    matched_concepts = []
    matched_keywords = []

    for concept, keywords in concept_keywords.items():
        hits = keyword_hits(text, keywords)
        if hits:
            matched_concepts.append(concept)
            matched_keywords.extend([f"{concept}:{kw}" for kw in hits])

    if len(matched_concepts) == 0:
        label = "正常"
    elif len(matched_concepts) == 1:
        label = matched_concepts[0]
    else:
        label = "多重"

    return label, matched_concepts, matched_keywords

classified = [classify_concepts(name) for name in work_names]

concept_viz_df = pd.DataFrame(
    {
        "社團名稱": work_names,
        "umap_x": coords[:, 0],
        "umap_y": coords[:, 1],
        "umap_z": coords[:, 2],
        "類別": [item[0] for item in classified],
        "命中概念": ["、".join(item[1]) for item in classified],
        "命中關鍵詞": ["、".join(item[2]) for item in classified],
    }
)

category_order = ["黃", "賭", "詐欺", "借貸", "多重", "正常"]
category_colors = {
    "黃": "#e83e8c",
    "賭": "#8e44ad",
    "詐欺": "#e67e22",
    "借貸": "#2ca02c",
    "多重": "#111827",
    "正常": "#9ca3af",
}

display(
    concept_viz_df["類別"]
    .value_counts()
    .reindex(category_order, fill_value=0)
    .rename_axis("類別")
    .to_frame("數量")
)

fig = px.scatter_3d(
    concept_viz_df,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="類別",
    category_orders={"類別": category_order},
    color_discrete_map=category_colors,
    hover_data={
        "社團名稱": True,
        "命中概念": True,
        "命中關鍵詞": True,
        "umap_x": False,
        "umap_y": False,
        "umap_z": False,
    },
    title="ckiplab/bert-base-chinese 社團名稱 Embedding：六類概念 UMAP 3D",
)
fig.update_traces(marker={"size": 3, "opacity": 0.78})
fig.update_layout(legend_title_text="類別", margin={"l": 0, "r": 0, "t": 48, "b": 0})
fig.show()

NameError: name 'work_names' is not defined

### 以抽樣 5000 惡意社團與 5000 一般社團 (二元標記) 繪製 UMAP 3D 視覺化圖

In [ ]:
# 根據 '類別' 欄位，從 `binary_viz_df` 中分別取出 '惡意社團' 和 '一般社團'
malicious_groups = binary_viz_df[binary_viz_df['類別'] == '惡意社團']
normal_groups = binary_viz_df[binary_viz_df['類別'] == '一般社團']

# 設定抽樣數量
sample_size = 5000

# 從惡意社團中抽樣，若數量不足則取全部
sampled_malicious_df = malicious_groups.sample(
    n=min(len(malicious_groups), sample_size), random_state=RANDOM_STATE
)

# 從一般社團中抽樣，若數量不足則取全部
sampled_normal_df = normal_groups.sample(
    n=min(len(normal_groups), sample_size), random_state=RANDOM_STATE
)

# 合併抽樣後的數據，用於二元分類的視覺化
sampled_binary_viz_df = pd.concat([sampled_malicious_df, sampled_normal_df]).reset_index(drop=True)

display(sampled_binary_viz_df["類別"].value_counts().rename_axis("類別").to_frame("數量"))

fig_sampled_binary = px.scatter_3d(
    sampled_binary_viz_df,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="類別",
    color_discrete_map={"惡意社團": "#d62728", "一般社團": "#1f77b4"},
    hover_data={
        "社團名稱": True, "命中關鍵詞": True,
        "umap_x": False, "umap_y": False, "umap_z": False
    },
    title=f"ckiplab/bert-base-chinese 社團名稱 Embedding：惡意 / 一般社團 UMAP 3D (各抽樣 {sample_size} 個)",
)
fig_sampled_binary.update_traces(marker={"size": 3, "opacity": 0.78})
fig_sampled_binary.update_layout(legend_title_text="類別", margin={"l": 0, "r": 0, "t": 48, "b": 0})
fig_sampled_binary.show()

### 以抽樣 5000 惡意社團與 5000 一般社團 (六類概念標記) 繪製 UMAP 3D 視覺化圖

In [ ]:
# 取得抽樣後的社團名稱列表
sampled_group_names_list = sampled_binary_viz_df['社團名稱'].tolist()

# 從 `concept_viz_df` 中篩選出這些抽樣的社團名稱
sampled_concept_viz_df = concept_viz_df[concept_viz_df['社團名稱'].isin(sampled_group_names_list)].reset_index(drop=True)

display(
    sampled_concept_viz_df["類別"]
    .value_counts()
    .reindex(category_order, fill_value=0)
    .rename_axis("類別")
    .to_frame("數量")
)

fig_sampled_concept = px.scatter_3d(
    sampled_concept_viz_df,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color="類別",
    category_orders={"類別": category_order},
    color_discrete_map=category_colors,
    hover_data={
        "社團名稱": True, "命中概念": True, "命中關鍵詞": True,
        "umap_x": False, "umap_y": False, "umap_z": False,
    },
    title=f"ckiplab/bert-base-chinese 社團名稱 Embedding：六類概念 UMAP 3D (基於前述抽樣 {sample_size} 個)",
)
fig_sampled_concept.update_traces(marker={"size": 3, "opacity": 0.78})
fig_sampled_concept.update_layout(legend_title_text="類別", margin={"l": 0, "r": 0, "t": 48, "b": 0})
fig_sampled_concept.show()